# Notebook 3 — DistilBERT Fine-Tuning
**What this notebook does:**
- Fine-tunes DistilBERT on Amazon Electronics training data (1999-2018)
- Evaluates on test data (2019-2023)
- Compares with LR and RF baselines
- Saves trained model for drift detection

⚠️ **NOTE:** This notebook takes 45-90 minutes on CPU. Run it and do other work.

In [ ]:
import sys
# Yeh command naye environment ke andar hi saari libraries install karegi
!"{sys.executable}" -m pip install pandas numpy matplotlib seaborn scikit-learn tqdm transformers
!"{sys.executable}" -m pip install torch --index-url https://download.pytorch.org/whl/cpu


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW  # Updated import
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('All imports loaded ✓')

c:\Users\Abdul Basit\Downloads\thesis_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
All imports loaded ✓


## Step 2 — Load Data

In [2]:
df_train = pd.read_csv('train_data.csv')
df_test  = pd.read_csv('test_data.csv')

print(f'Training data: {len(df_train):,} reviews')
print(f'Test data: {len(df_test):,} reviews')

# Use subset for speed on CPU
# Increase if you have more time/resources
TRAIN_SAMPLES = 15000
TEST_SAMPLES  = 5000

train_subset = df_train.sample(n=min(TRAIN_SAMPLES, len(df_train)), random_state=42)
test_subset  = df_test.sample(n=min(TEST_SAMPLES, len(df_test)), random_state=42)

print(f'\nUsing {len(train_subset):,} training samples')
print(f'Using {len(test_subset):,} test samples')

Training data: 86,853 reviews
Test data: 106,511 reviews

Using 15,000 training samples
Using 5,000 test samples


## Step 3 — Tokenizer & Dataset Class

In [3]:
print('Loading DistilBERT tokenizer...')
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class AmazonReviewDataset(Dataset):
    def __init__(self, texts, labels, max_len=128):
        self.texts   = list(texts)
        self.labels  = list(labels)
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets
train_dataset = AmazonReviewDataset(
    train_subset['text'].fillna('').values,
    train_subset['label'].values
)
test_dataset = AmazonReviewDataset(
    test_subset['text'].fillna('').values,
    test_subset['label'].values
)

# DataLoaders
BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')
print('Dataset ready ✓')

Loading DistilBERT tokenizer...
Train batches: 938
Test batches:  313
Dataset ready ✓


In [4]:
print("Training period:")
print(train_subset['year'].min(), "-", train_subset['year'].max())

print("\nTesting period:")
print(test_subset['year'].min(), "-", test_subset['year'].max())

Training period:
2001 - 2018

Testing period:
2019 - 2023


## Step 4 — Load DistilBERT Model

In [5]:
print('Loading DistilBERT model (downloading ~270MB)...')
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3  # Negative, Neutral, Positive
)
model = model.to(device)

# Optimizer & Scheduler
EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

print(f'Model loaded ✓')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Training for {EPOCHS} epochs on {device}')

Loading DistilBERT model (downloading ~270MB)...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded ✓
Total parameters: 66,955,779
Training for 3 epochs on cpu


## Step 5 — Training Functions

In [6]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc='Training'):
        optimizer.zero_grad()

        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


def evaluate_model(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating'):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            preds = outputs.logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    return all_preds, all_labels, acc, f1

print('Training functions defined ✓')

Training functions defined ✓


## Step 6 — Fine-Tune DistilBERT (Main Training)
⚠️ This takes 45-90 minutes on CPU. Please wait.

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}

print('Starting DistilBERT Fine-Tuning...')
print('Training data: Amazon Electronics Reviews (1999-2018)')
print('='*55)

for epoch in range(EPOCHS):
    print(f'\nEPOCH {epoch+1}/{EPOCHS}')
    print('-'*40)

    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler)

    # Evaluate
    preds, labels, val_acc, val_f1 = evaluate_model(model, test_loader)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%}')
    print(f'Val Acc:    {val_acc:.2%}   | Val F1:   {val_f1:.4f}')

print('\n' + '='*55)
print('FINE-TUNING COMPLETE!')
print(f'Final Accuracy: {history["val_acc"][-1]:.2%}')
print(f'Final F1:       {history["val_f1"][-1]:.4f}')

Starting DistilBERT Fine-Tuning...
Training data: Amazon Electronics Reviews (1999-2018)

EPOCH 1/3
----------------------------------------


Training:  87%|████████▋ | 818/938 [1:43:24<14:47,  7.40s/it]

## Step 7 — Final Evaluation & Classification Report

In [ ]:
print('DISTILBERT FINAL RESULTS')
print('='*55)
print(classification_report(
    labels, preds,
    target_names=['Negative', 'Neutral', 'Positive']
))

## Step 8 — Training Curves Graph

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(range(1, EPOCHS+1), history['train_loss'],
             marker='o', color='#e74c3c', linewidth=2)
axes[0].set_title('Training Loss per Epoch', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(range(1, EPOCHS+1), history['val_acc'],
             marker='s', color='#2E75B6', linewidth=2, label='Accuracy')
axes[1].plot(range(1, EPOCHS+1), history['val_f1'],
             marker='^', color='#2ecc71', linewidth=2, label='F1 Score')
axes[1].set_title('Validation Accuracy & F1 per Epoch', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('DistilBERT Fine-Tuning — Amazon Electronics Reviews', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/03_distilbert_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved ✓')

## Step 9 — Confusion Matrix

In [ ]:
plt.figure(figsize=(7, 5))
cm = confusion_matrix(labels, preds)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Negative','Neutral','Positive'],
    yticklabels=['Negative','Neutral','Positive']
)
plt.title(f'DistilBERT Confusion Matrix\nAccuracy: {history["val_acc"][-1]:.2%}',
          fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../outputs/03_distilbert_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved ✓')

## Step 10 — Save Model & Final Comparison

In [ ]:
# Save DistilBERT
model.save_pretrained('../models/distilbert_amazon')
tokenizer.save_pretrained('../models/distilbert_amazon')
print('DistilBERT saved to models/distilbert_amazon/ ✓')

# Final Comparison Table
print('\n' + '='*65)
print('FINAL MODEL COMPARISON')
print('='*65)
print(f'{"Model":<25} {"Accuracy":<12} {"Drift":<10} {"SHAP":<10}')
print('-'*65)
print(f'{"Logistic Regression":<25} {"86%":<12} {"❌":<10} {"❌":<10}')
print(f'{"Random Forest":<25} {"85%":<12} {"❌":<10} {"❌":<10}')
print(f'{"DistilBERT (Ours)":<25} {str(round(history["val_acc"][-1]*100,1))+"%":<12} {"✅":<10} {"✅":<10}')
print('='*65)
print('\n✅ Notebook 3 Complete! Run Notebook 4 next.')